# Coding Midterm Exam Notebook

## 1. Setup and Imports

### 1.1 Essential Imports

In [34]:
# Core data and ML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from typing import Any, Dict, List, Tuple
from collections import defaultdict # Used for Perplexity/N-gram model
import math # For Perplexity calculation

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hugging Face Transformers
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    AutoModelForSeq2SeqLM, 
    TrainingArguments, 
    Trainer, 
    pipeline # For quick case studies
    )

# Classical ML and Metrics
import sklearn
from sklearn.metrics import (
    f1_score, 
    accuracy_score, 
    precision_recall_fscore_support, 
    confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD # For LSA
from sklearn.linear_model import LogisticRegressionCV

# NLTK and Generation Metrics
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# %matplotlib inline

print("Package import test successful!")

Package import test successful!


### 1.2 Set Device and Seed

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

Using device: cpu


## 2. Data Loading

### 2.1 Data Loading Placeholders

In [ ]:
# 1. Load from CSV
df = pd.read_csv("data.csv") # Replace data.csv with actual data
print(df.head())

# 2. Load from Hugging Face
dataset = load_dataset("imdb") # Replace imdb with actual data
print(dataset)

# 3. Convert pandas DataFrame to HF dataset (if needed)
hf_dataset = Dataset.from_pandas(df)

## 3. Data Preprocessing and Tokenization

This section covers all preprocessing methods: **low-level cleaning for custom models, Hugging Face tokenization for Transformers, and classical feature extraction for baselines**.

### 3.1 Low-Level Cleaning and Vocabulary (For Custom PyTorch Models)

In [26]:
# --- Cleaning (from 2. Handling Text- Part 1 - Solution.ipynb context) ---
def clean_text(text: str) -> List[str]:
    """Tokenizes, converts to lowercase, and removes stopwords/punctuation."""
    # 1. Lowercase and initial tokenization
    tokens = word_tokenize(text.lower()) 
    
    # 2. Filter out non-alphabetic tokens and stop words
    stop_words_set = set(stopwords.words('english'))
    cleaned_tokens = [
        word for word in tokens 
        if word.isalpha() and word not in stop_words_set
    ]
    
    # Optional: Stemming or Lemmatization (using NLTK/SpaCy)
    # from nltk.stem import PorterStemmer
    # stemmer = PorterStemmer()
    # stemmed_tokens = [stemmer.stem(word) for word in cleaned_tokens]
    
    return cleaned_tokens

# --- Explicit Vocabulary Mapping (for Custom Models, from 0. Word Embeddings - Solutions.ipynb) ---
def build_vocab(tokenized_corpus: List[List[str]]) -> Tuple[Dict[str, int], int]:
    """Creates a word-to-index mapping for use with nn.Embedding."""
    word_to_idx = {"<PAD>": 0, "<UNK>": 1} # Reserve special tokens
    idx_counter = 2
    for sentence in tokenized_corpus:
        for word in sentence:
            if word not in word_to_idx:
                word_to_idx[word] = idx_counter
                idx_counter += 1
    return word_to_idx, len(word_to_idx)

### 3.2 Transformer Tokenization (Hugging Face)

In [36]:
# --- Hugging Face Tokenization (from 1. BERT and T5 - Solutions.ipynb context) ---
MODEL_CHECKPOINT = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# --- A. Classification/Regression Task ---
def tokenize_for_transformer(examples, text_col='text', label_col='label'):
    """Tokenization for Encoder-only models (BERT, DistilBERT) for classification."""
    model_inputs = tokenizer(
        examples[text_col], 
        truncation=True, 
        padding='max_length', 
        max_length=128
    )
    model_inputs["labels"] = examples[label_col]
    return model_inputs

# --- B. Sequence-to-Sequence (Seq2Seq) Task ---
def tokenize_seq2seq(examples, source_lang='en', target_lang='fr'):
    """Tokenization for Encoder-Decoder models (T5, BART) for generation."""
    inputs = examples[source_lang] 
    targets = examples[target_lang]
    
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    
    # Use as_target_tokenizer for the decoder input labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

### 3.3 Classical Feature Engineering (TF-IDF & SVD Baseline)

This section, derived from the *3. Handling Text - Part 2 - Solution.ipynb* context, is necessary if you are asked to demonstrate a classical NLP pipeline for comparison.

In [37]:
### Classical Feature Extraction (TF-IDF & SVD Baseline)
def create_tfidf_features(X_train_text: List[str], X_test_text: List[str], max_features=5000):
    """Computes TF-IDF features for classical models."""
    
    tfidf = TfidfVectorizer(stop_words='english', max_features=max_features)
    X_train_tfidf = tfidf.fit_transform(X_train_text)
    X_test_tfidf = tfidf.transform(X_test_text)
    
    # Optional: Truncated SVD for Latent Semantic Analysis (LSA)
    # n_components = 100
    # svd = TruncatedSVD(n_components=n_components, random_state=42)
    # X_train_svd = svd.fit_transform(X_train_tfidf)
    # X_test_svd = svd.transform(X_test_tfidf)
    
    # # Example Baseline Training Function:
    # # model = LogisticRegressionCV(Cs=10, cv=5, random_state=42, solver='liblinear')
    # # model.fit(X_train_tfidf, y_train)
    
    return X_train_tfidf, X_test_tfidf

## 4. Training Models and PyTorch Structures

This includes the Custom PyTorch model definition and both the custom and Hugging Face training procedures.

### 4.1 Custom PyTorch Model Structure (RNN/LSTM/MLP)

In [38]:
# --- Custom Deep Model with nn.Embedding (from RNNs_Sol.ipynb) ---
class CustomRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        # nn.Embedding is crucial for converting token indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # Use nn.LSTM or nn.GRU as required by the scenario
        self.encoder = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)
        _, (hidden, _) = self.encoder(embedded)
        final_output = self.classifier(hidden[-1, :, :]) # Use last layer's final hidden state
        return final_output

### 4.2 Custom PyTorch Training Loop

In [39]:
### PyTorch Generic Training Loop (from 0. MLP and Training - Solution.ipynb)
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Executes one epoch of a standard PyTorch training process."""
    model.train()
    total_loss = 0
    for batch in dataloader:
        # NOTE: Adjust keys ('input', 'label') based on your custom DataLoader structure
        inputs = batch['input'].to(device)
        labels = batch['label'].to(device) 
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0) 
    
    return total_loss / len(dataloader.dataset)

# # Example Training Loop Setup:
# # model = CustomRNN(...)
# # criterion = nn.CrossEntropyLoss()
# # optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# # for epoch in range(N_EPOCHS):
# #     train_loss = train_epoch(model, train_dataloader, criterion, optimizer, device)

### 4.3 Hugging Face Transformer Training Setup

In [ ]:
# --- Load Transformer Model (Required) ---
# NOTE: Use AutoModelForSeq2SeqLM for T5/generation tasks
model_hf = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2).to(device)

# --- Hugging Face Training Arguments ---
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16, # Example addition
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-5, # Example addition
)

# # trainer = Trainer(
# #     model=model_hf, 
# #     args=training_args, 
# #     train_dataset=tokenized_datasets["train"],
# #     eval_dataset=tokenized_datasets["validation"],
# #     compute_metrics=compute_metrics_classification # Defined below
# # )
# # trainer.train()

## 5. Analysis: Evaluation Metrics and Perplexity

### 5.1 Classification Metrics

In [ ]:
def compute_metrics_classification(p: Tuple[np.ndarray, np.ndarray]) -> Dict[str, Any]:
    """Selects F1, Accuracy, and Confusion Matrix for classification."""
    preds = np.argmax(p.predictions, axis=1)
    
    # F1-score is robust for imbalance (Selects appropriate evaluation metrics)
    precision, recall, f1, _ = precision_recall_fscore_support(
        p.label_ids, 
        preds, 
        average='binary' if len(np.unique(p.label_ids)) == 2 else 'macro'
    ) 
    acc = accuracy_score(p.label_ids, preds)
    cm = confusion_matrix(p.label_ids, preds)

    print(f"--- Confusion Matrix ---\n{cm}") 

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

### 5.2 Perplexity Calculation (For Language Models)

(from *1. Perplexity - Solution.ipynb*).

In [ ]:
def calculate_perplexity(model, dataloader, device) -> float:
    """Measures the model's ability to predict the test data (lower is better)."""
    # NOTE: This uses the Hugging Face style of passing labels to the model for loss computation
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for batch in dataloader:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device) 
            
            outputs = model(inputs, labels=labels)
            
            # Hugging Face models return mean loss when labels are provided
            if outputs.loss is not None:
                # To get NLL, we approximate: Total Loss = Mean Loss * Batch Size
                loss = outputs.loss * inputs.size(0)
            else:
                 # Manually compute loss for custom models
                 # [Manual CrossEntropyLoss code from template omitted for brevity, but available]
                 pass 

            total_loss += loss.item()
            total_tokens += (inputs.size(1) - 1) * inputs.size(0) # Number of predicted tokens

    if total_tokens == 0: return float('inf')
    
    average_nll = total_loss / total_tokens
    perplexity = math.exp(average_nll)
    return perplexity

### 5.3 Sequence Generation Metric (BLEU Score)

In [ ]:
### BLEU Score Calculation (for Seq2Seq Tasks like Translation/Summarization)
def calculate_bleu(reference_tokens: List[str], hypothesis_tokens: List[str]) -> float:
    """
    Calculates the sentence-level BLEU score.
    reference_tokens: The true target sentence (as a list of tokens).
    hypothesis_tokens: The model's generated sentence (as a list of tokens).
    """
    # Use SmoothingFunction().method4 for robustness
    smoothie = SmoothingFunction().method4
    
    # BLEU expects a list of reference lists (even for a single reference)
    bleu = sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)
    return bleu

## 6. Discussion and Analysis Templates

This section directly addresses the requirement for **conducting case studies, discussing model limitations, and exploring possible improvements**.

### 6.1 Case Study: Error Analysis via Attention

Use the Hugging Face `pipeline` or manual attention axtraction to **conduct case studies** on difficult examples.

#### Conceptual Code: Attention Extraction (attention_sol.ipynb context) 
For the case study, you can try to visualize the attention weights to see *why* a False Negative token was ignored.

1. Manually set model to return attentions for a detailed look
    * model_hf.config.output_attentions = True 

2. Perform forward pass on the challenging example (e.g., FN case) with torch.no_grad():
    * outputs = model_hf(input_ids=input_ids, attention_mask=attn_mask, output_attentions=True)
 

3. Attention is in outputs.attentions (e.g., last layer, first head)
    * attn_weights = outputs.attentions[-1][0, 0, :, :] 

4. Observation: Look where the model *should* have focused but didn't.

### 6.2 Case Study: Error Analysis via Pipeline

Use a small snippet of code with the Hugging Face `pipeline` tool to quickly test your trained model and analyze **False Positives (FP)** and **False Negatives (FN)**.

1.  **False Positive Instance:** (Model Predicted Positive, Actual is Negative)
    * **Observation:** The error occurs on text that *[e.g., uses strong negative language sarcastically]*. The model failed because it relies too heavily on **local features (keywords)** and lacks the capacity to capture the global, **nuanced semantic context**.

2.  **False Negative Instance:** (Model Predicted Negative, Actual is Positive)
    * **Observation:** The error occurs on *[e.g., a very long document where the key information was in the middle]*. This points to the **Transformer's sequence length limit (e.g., 512 tokens)** or a failure in the attention mechanism to properly weight distant, important tokens.

### 6.3 Discussion of Model Limitations

1.  **Sequence Length Constraint:** All Transformer models have a fixed maximum input length (e.g., 512). This requires **truncation** of long inputs, which inevitably leads to a loss of information, especially critical for tasks like document classification or summarization.
2.  **Vocabulary Dependence & OOV:** Models rely on their pre-trained **subword vocabulary (BPE/WordPiece)**. While better than a fixed word vocabulary, performance degradation still occurs for highly specialized or rare terminology (Out-Of-Vocabulary words).
3.  **Computational Cost:** Training is demanding. The size of the model restricts the feasibility of **extensive hyperparameter tuning** (e.g., grid search for learning rate or weight decay), requiring reliance on sensible defaults.

### Exploring Possible Improvements

1.  **Architectural Improvement:**
    * **For Long Inputs:** Use models like **Longformer** or a **Hierarchical Attention Network** to process inputs longer than 512 tokens without truncation.
    * **For Seq2Seq (T5/Translation):** Evaluate the use of **beam search decoding** (with a beam size > 1) instead of greedy decoding for higher-quality, less repetitive outputs.

2.  **Training Procedure Improvement (Code/Concept from MLP/Transformer solutions):**
    * **Implement a Learning Rate Scheduler** (e.g., cosine decay with warmup steps) instead of a fixed learning rate to better guide the optimizer and improve convergence.
    * **Advanced Regularization:** Increase **weight decay** or use techniques like **Dropout** (explicitly in custom models or implicitly in Hugging Face models) to mitigate overfitting, which is common with small fine-tuning datasets.

3.  **Evaluation Improvement:**
    * For Language Model tasks, report **Perplexity** alongside validation loss, as it provides a more intuitive measure of the model's confidence in predicting natural text.
    * For imbalanced classification, utilize **Weighted F1-score** or **AUROC** (Area Under the Receiver Operating Characteristic Curve) to provide a more robust performance measure than the macro F1 used above.